# YouTube 댓글 데이터 전처리 및 기초 정제 파이프라인

이 노트북은 연구 논문 수준의 데이터 분석을 위해 YouTube 댓글 데이터를 로드하고, 중복치 및 노이즈(URL, 특수문자, 닉네임, 타임스탬프 등)를 정제하는 첫 번째 단계입니다.

## 1. 환경 설정 및 라이브러리 로드

In [1]:
import pandas as pd
import re
import glob
import os

# 결과 확인을 위한 설정
pd.set_option('display.max_colwidth', None)

## 2. 데이터 로드 및 통합

- raw_data 폴더 내의 모든 csv 파일을 읽어와 하나로 통합합니다.

In [2]:
# raw_data 폴더 내의 모든 csv 파일 읽기
raw_path = os.path.join("..", "data", "raw")
file_list = glob.glob(os.path.join(raw_path, "*.csv"))

df_list = []
for file in file_list:
    temp_df = pd.read_csv(file, encoding='utf-8-sig', on_bad_lines='skip', engine='python')
    df_list.append(temp_df)

if df_list:
    df = pd.concat(df_list, ignore_index=True)
else:
    print("파일을 찾을 수 없습니다.")
    exit()

print(f"총 댓글 수: {len(df)}")

총 댓글 수: 79239


## 3. 데이터 결측치/중복치 처리 및 텍스트 정제

- 중복 데이터 및 결측치 제거
- 텍스트 내 불필요한 노이즈(@닉네임, 타임스탬프, URL 등) 제거
- 유튜브 특유의 반복되는 자음/모음 정규화 및 한글만 남기기

In [5]:
def load_and_basic_clean(df):
    initial_count = len(df)
    df = df.dropna(subset=['text'])
    df = df.drop_duplicates(subset=['text'])
    
    print(f"Initial data (successfully parsed): {initial_count}")
    print(f"After removing NaNs and Duplicates: {len(df)}")
    return df

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@[^\s]+', '', text)
    text = re.sub(r'\d{1,2}:\d{2}(?::\d{2})?', '', text)
    text = re.sub(r'[^가-힣\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = load_and_basic_clean(df)
df['cleaned_text'] = df['text'].apply(clean_text)
print("텍스트 정제 완료")
display(df[['text', 'cleaned_text']].head())

Initial data (successfully parsed): 74356
After removing NaNs and Duplicates: 74356
텍스트 정제 완료


,text,cleaned_text
0,정말 저런 남편이 있기나한건지......?\n나두 우울증.....\n참 부럽다!!!!,정말 저런 남편이 있기나한건지 나두 우울증 참 부럽다
1,제목: 남편이 매일 느끼던 감정을 이제 아내가 느끼기 시작했다.,제목 남편이 매일 느끼던 감정을 이제 아내가 느끼기 시작했다
2,우울증이 있는 가족의삶 공감 못함 고통 무너짐,우울증이 있는 가족의삶 공감 못함 고통 무너짐
3,남자의 우울증은 약한사람 이라는 인식 없어졌으면 좋겠다,남자의 우울증은 약한사람 이라는 인식 없어졌으면 좋겠다
4,한국여자가 머한다고 우울증에 걸리냐! 바쁘게 지내봐! 커피숍다니면서 노가리까지나말고 시장가서 폐지줍는 노인들도보고 채소파는 노인들도 보고 좀 나가서 쳐 걷고! 커피숍에 모여서 비싼커피마시면서 우아함이나 뽐내지말고! 진심 여자 우울증은 사치야,한국여자가 머한다고 우울증에 걸리냐 바쁘게 지내봐 커피숍다니면서 노가리까지나말고 시장가서 폐지줍는 노인들도보고 채소파는 노인들도 보고 좀 나가서 쳐 걷고 커피숍에 모여서 비싼커피마시면서 우아함이나 뽐내지말고 진심 여자 우울증은 사치야


## 4. 정제된 중간 데이터 저장

- 토큰화를 진행하기 전, 정제가 완료된 텍스트 데이터를 임시 저장합니다.

In [6]:
output_dir = os.path.join("..", "data", "processed")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "토크나이징_전_전처리.csv")
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"정제 데이터 저장 완료: {output_path}")

정제 데이터 저장 완료: preprocessed_comments.csv
